# Numerical integration

Three quadrature rules for approximating ∫_a^b f(x) dx.

| Rule | Points | Error order | Notes |
|---|---|---|---|
| Trapezoid | n+1 equally spaced | O(h^2) | Simple, general |
| Simpson | n+1 equally spaced (n even) | O(h^4) | Exact for cubics |
| Gauss-Legendre | n Gauss points | O(h^{2n}) | Exact for degree 2n-1 polynomials |

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import matplotlib.pyplot as plt

from src.integration import trapezoid, simpson, gauss_legendre

## Trapezoid rule

Approximates the integral by summing trapezoids:

    T_n = h/2 * [f(a) + 2f(x_1) + ... + 2f(x_{n-1}) + f(b)],  h = (b-a)/n

Error: -(b-a)h^2/12 * f''(ξ) for some ξ ∈ [a,b].

In [ ]:
# Integrate sin(x) from 0 to pi; exact = 2.0
exact = 2.0
ns = [4, 8, 16, 32, 64, 128, 256]
trap_errors = [abs(trapezoid(math.sin, 0, math.pi, n) - exact) for n in ns]

plt.loglog(ns, trap_errors, marker='o', label='trapezoid')
plt.loglog(ns, [1/n**2 for n in ns], '--', label='O(h^2) reference')
plt.xlabel('n')
plt.ylabel('|error|')
plt.title('Trapezoid error on ∫sin(x)dx, 0 to π')
plt.legend()
plt.tight_layout()
plt.show()

## Simpson's rule

Uses parabolas instead of straight lines between consecutive points:

    S_n = h/3 * [f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + ... + f(x_n)],  n even

Error: -(b-a)h^4/180 * f''''(ξ). Two orders better than trapezoid for smooth f.

In [ ]:
simp_errors = [abs(simpson(math.sin, 0, math.pi, n) - exact) for n in ns]

plt.loglog(ns, trap_errors, marker='o', label='trapezoid')
plt.loglog(ns, simp_errors, marker='s', label='simpson')
plt.loglog(ns, [1/n**2 for n in ns], '--', label='O(h^2)')
plt.loglog(ns, [1/n**4 for n in ns], ':', label='O(h^4)')
plt.xlabel('n')
plt.ylabel('|error|')
plt.title('Trapezoid vs Simpson on ∫sin(x)dx, 0 to π')
plt.legend()
plt.tight_layout()
plt.show()

## Gauss-Legendre quadrature

Chooses both the quadrature points and weights optimally. With n points, the rule is exact for all polynomials of degree ≤ 2n-1.

Points are roots of the Legendre polynomial P_n(x); weights are determined by requiring exactness on 1, x, ..., x^{2n-1}.

The rule integrates over [-1, 1] by default; the implementation shifts and scales to [a, b].

In [ ]:
# Compare all three at fixed n=8 on several integrals
cases = [
    ('sin(x), [0,π]',   math.sin,  0,        math.pi,  2.0),
    ('exp(x), [0,1]',   math.exp,  0,        1.0,      math.e - 1),
    ('1/x, [1,e]',      lambda x: 1/x, 1.0,  math.e,   1.0),
    ('x^5, [0,1]',      lambda x: x**5, 0, 1.0, 1/6),
]

print(f'{'integrand':<20} {'trap':>12} {'simpson':>12} {'gauss':>12}')
print('-' * 60)
for name, f, a, b, exact in cases:
    e_t = abs(trapezoid(f, a, b, 8) - exact)
    e_s = abs(simpson(f, a, b, 8)   - exact)
    e_g = abs(gauss_legendre(f, a, b, 8) - exact)
    print(f'{name:<20} {e_t:>12.2e} {e_s:>12.2e} {e_g:>12.2e}')

## Gauss-Legendre convergence

For analytic (infinitely differentiable) integrands, Gauss-Legendre achieves spectral convergence — error decays exponentially with n rather than as a power of h.

In [ ]:
ns_gl = list(range(1, 16))
gl_errors = [abs(gauss_legendre(math.sin, 0, math.pi, n) - 2.0) for n in ns_gl]

plt.semilogy(ns_gl, gl_errors, marker='o')
plt.xlabel('n (number of quadrature points)')
plt.ylabel('|error|')
plt.title('Gauss-Legendre convergence on ∫sin(x)dx')
plt.tight_layout()
plt.show()